# Tostal — Train & Evaluate All Models

This notebook clones (or pulls) the repo, installs dependencies, and runs:
1. **Facies classifier** — XGBoost + spatial features on FORCE 2020 + Taranaki well logs
2. **DINOv2 classifier** — vit_s14 backbone on DCID drill core images
3. **Thin section classifier** — DINOv2 + LITHOS petrography dataset (25 mineral classes)
4. **Synthetic core** — core property prediction from well logs (lithology, porosity, perm)
5. **Evaluate** — accuracy, F1, confusion matrix, top-k, cross-basin

Estimated runtime: ~3 min CPU (facies) + ~3 min GPU (DINOv2) + ~7 min GPU (thin section)

In [ ]:
import os, sys, subprocess
from pathlib import Path

## 1. Clone or Pull the Repository

In [ ]:
REPO_URL = "https://github.com/beaglabs/tostal.git"

repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists():
        break
    repo_root = repo_root.parent

if (repo_root / ".git").exists():
    print(f"Found existing repo at {repo_root}, pulling latest...")
    subprocess.run(["git", "pull"], cwd=repo_root, check=True)
else:
    repo_root = Path.home() / "tostal"
    if (repo_root / ".git").exists():
        print(f"Repo already exists at {repo_root}, pulling latest...")
        subprocess.run(["git", "pull"], cwd=repo_root, check=True)
    elif repo_root.exists():
        print(f"Directory {repo_root} exists but is not a git repo, removing...")
        import shutil
        shutil.rmtree(repo_root)
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    else:
        print(f"Cloning into {repo_root}...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)

print(f"Repo root: {repo_root}")

## 2. Install Dependencies

In [ ]:
requirements = repo_root / "api" / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
else:
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "xgboost", "pykrige", "scikit-learn", "torch", "torchvision",
        "scipy", "datasets", "lasio", "openpyxl", "tqdm", "pillow",
        "kagglehub", "mlcroissant",
    ], check=True)

print("Dependencies installed.")

## 3. Train Facies Classifier (XGBoost + Spatial Features)

Downloads FORCE 2020 well logs with lithostratigraphy, engineers spatial features
(normalized depth, rolling-window stats, neighboring values), and trains an XGBoost
classifier. Saves to `training/models/xgboost_facies.json`.

In [ ]:
%cd {repo_root}

import shutil
data_dir = Path("data/force2020")
processed_pt = data_dir / "processed.pt"
las_dir = data_dir / "las"
if processed_pt.exists():
    processed_pt.unlink()
if las_dir.exists():
    shutil.rmtree(las_dir)

subprocess.run([sys.executable, "training/scripts/train_xgboost_facies.py"], check=True)

## 3b. Train on Taranaki Basin (Cross-Basin)

Downloads the IBM-curated Taranaki Basin well log dataset from Zenodo and trains
a cross-basin facies classifier. Combines with FORCE 2020 for multi-basin training.

In [ ]:
%cd {repo_root}

# Import and run Taranaki processing
sys.path.insert(0, str(repo_root))
from training.data import process_taranaki, FACIES_CLASSES, build_spatial_features, TARANAKI_LABEL_NAMES
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Load Taranaki data
try:
    taranaki = process_taranaki(cache_dir="data/taranaki")
    print(f"Taranaki: {len(taranaki)} wells")

    # Build dataset
    X_list, y_list = [], []
    for name, data in taranaki.items():
        curves = data["well_log"].numpy()
        facies = data.get("facies")
        if facies is None:
            continue
        facies = facies.numpy()
        if curves.shape != (6, 512):
            continue
        X_spatial = build_spatial_features(curves, 512)
        for d in range(512):
            label = facies[d]
            if label < 0:
                continue
            X_list.append(X_spatial[d])
            y_list.append(label)

    if len(X_list) > 0:
        X_t = np.array(X_list, dtype=np.float32)
        y_t = np.array(y_list, dtype=np.int64)
        print(f"  {len(X_t)} labeled points from Taranaki")
        print(f"  Label map: {TARANAKI_LABEL_NAMES}")

        # Train Taranaki-only model
        X_tr, X_te, y_tr, y_te = train_test_split(X_t, y_t, test_size=0.2, random_state=42)
        t_model = xgb.XGBClassifier(n_estimators=100, max_depth=6, eval_metric="mlogloss")
        t_model.fit(X_tr, y_tr, verbose=False)
        y_pred = t_model.predict(X_te)
        print(f"  Taranaki test accuracy: {accuracy_score(y_te, y_pred):.4f}")
        print(f"  Taranaki test F1 (wtd): {f1_score(y_te, y_pred, average='weighted'):.4f}")
        print(f"\n  Per-class F1 (local idx → FACIES name):")
        f1_per_class = f1_score(y_te, y_pred, average=None, labels=sorted(TARANAKI_LABEL_NAMES.keys()))
        for local_idx, f1_val in zip(sorted(TARANAKI_LABEL_NAMES.keys()), f1_per_class):
            print(f"    {local_idx} ({TARANAKI_LABEL_NAMES[local_idx]:20s}): {f1_val:.4f}")
    else:
        print("  No labeled points found in Taranaki")
except Exception as e:
    print(f"Taranaki processing skipped: {e}")

## 4. Train DINOv2 Classifier (Vit-S/14) — Core Photo Analysis

Loads 1000 DCID drill core images, extracts features with frozen DINOv2 vit_s14
backbone, and trains a lightweight classifier head for core photo lithology.
Saves to `training/models/dino_classifier.pt`.

**~2 min on GPU, ~10 min on CPU.**

In [ ]:
%cd {repo_root}
subprocess.run([sys.executable, "training/scripts/train_dino_classifier.py"], check=True)

## 4b. Train Thin Section Petrography Classifier (DINOv2) — LITHOS Dataset

Downloads the LITHOS dataset via kagglehub (Paola Ruiz Puentes, NeurIPS 2025),
loads Croissant metadata, extracts DINOv2 vit_s14 features from polarized-light
petrography patches, and trains a 25-class mineral classifier head.

**Requires Kaggle API credentials** (`~/.kaggle/kaggle.json`). Accept dataset
terms at https://www.kaggle.com/datasets/paolaruizpuentes/lithos-dataset

**~5 min feature extraction + ~2 min training on GPU. ~15 min on CPU.**

In [ ]:
%cd {repo_root}
subprocess.run([sys.executable, "training/scripts/train_thin_section.py"], check=True)

## 5. Synthetic Core Demo

Uses the facies classifier for lithology + empirical petrophysical transforms
to predict porosity and permeability from well logs. Full synthetic core model
(logs → core properties) is trained when NPD core description data is available.

Demonstrates the per-depth confidence and flags API output.

In [ ]:
%cd {repo_root}

sys.path.insert(0, str(repo_root))
from api.app.services.classifier_model import classify_synthetic_core
from training.data import process_force2020
import json

# Load a test well
wells = process_force2020(cache_dir="data/force2020")
sample_well = list(wells.values())[0]
curves = sample_well["well_log"].numpy()

# Run synthetic core prediction
result = classify_synthetic_core(curves)

# Show output structure
print(f"Lithology classes: {result['classes']}")
print(f"Outputs: {list(result['outputs'].keys())}")
print(f"Porosity range: {result['outputs']['porosity']['range']}")
print(f"Permeability range: {result['outputs']['permeability']['range']}")

# Show flags (low confidence zones, suggested coring intervals)
flags = result.get('flags', {})
print(f"\nLow confidence zones: {len(flags.get('low_confidence_zones', []))}")
print(f"Suggested coring intervals: {len(flags.get('suggested_coring_intervals', []))}")
for interval in flags.get('suggested_coring_intervals', [])[:3]:
    print(f"  Depth {interval['from']}-{interval['to']}")

print("\nSynthetic core demo complete.")

## 6. Evaluate Models

XGBoost: accuracy, F1, per-class F1, confusion matrix on FORCE 2020 + Taranaki.
DINOv2: top-1 and top-3 accuracy on held-out DCID subset.
Thin section: top-1 and top-3 accuracy on held-out LITHOS subset.

In [ ]:
%cd {repo_root}
subprocess.run([sys.executable, "training/scripts/evaluate.py"], check=True)

## 7. Verify Outputs

In [ ]:
models_dir = repo_root / "training" / "models"
for f in sorted(models_dir.glob("*")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:40s}  {size_mb:8.2f} MB")
print("\nDone.")